In [3]:
import g2_lib as g2


In [51]:
import g2_lib as g2

bandwidth = 900
num_nodes = 8

# Create a ring network with 8 nodes
ring_network = g2.Network([])
links = []
for i in range(1, num_nodes + 1):
    src = str(i)
    dst = str(i + 1) if i < num_nodes else "1"
    link_name = f"{src}-{dst}"
    links.append(link_name)
    ring_network = ring_network.add_link(g2.g2_lib_compiled_bindings.Link(src, dst, link_name, bandwidth))
    link_name = f"{dst}-{src}"
    links.append(link_name)
    ring_network = ring_network.add_link(g2.g2_lib_compiled_bindings.Link(dst, src, link_name, bandwidth))



# Add flowgroups with unique names
# Long path around the ring from 1 to 5 (1-2-3-4-5)
long_path_links = ["1-2", "2-3", "3-4", "4-5"]
ring_network2 = ring_network.add_flowgroup(long_path_links, "ring_long_1_5", 1)

print("Ring Network with 8 nodes and a shortcut (1-5)")
route_long = ring_network2.get_route_result_json("ring_long_1_4", "1", "4", 1, num_routes=10)
print("Route for ring_long_1_4 (1-2-3-4):", route_long['0']['path'])
ring_network2 = ring_network2.add_flowgroup(route_long['0']['path'], "ring_long_1_4", 1)
for f in ring_network2.flow_names():
    print(f"Flowgroup: {f}, Flow rate: {ring_network2.get_flow_rate(f)}")

# Add another flowgroup for a long path in the opposite direction (1-8-7-6-5)
print("\nRing Network with 8 nodes and a shortcut (1-5)")
route_long = ring_network.get_route_result_json("ring_long_1_4", "1", "4", 1, num_routes=10)
print("Route for ring_long_1_4 (1-2-3-4):", route_long['0']['path'])
ring_network3 = ring_network.add_flowgroup(route_long['0']['path'], "ring_long_1_4", 1)
for f in ring_network3.flow_names():
    print(f"Flowgroup: {f}, Flow rate: {ring_network3.get_flow_rate(f)}")

# Add another flowgroup for a long path in the opposite direction (1-8-7-6-5)
ring_network4 = ring_network.add_flowgroup(long_path_links, "ring_long_1_5", 1)
normal_path_links = ["1-2", "2-3", "3-4"]
print("\nRing Network with 8 nodes and a shortcut (1-5)")
ring_network4 = ring_network4.add_flowgroup(normal_path_links, "ring_long_1_4", 1)
for f in ring_network4.flow_names():
    print(f"Flowgroup: {f}, Flow rate: {ring_network4.get_flow_rate(f)}")

Ring Network with 8 nodes and a shortcut (1-5)
Route for ring_long_1_4 (1-2-3-4): ['1-8', '8-7', '7-6', '6-5', '5-4']
Flowgroup: ring_long_1_5, Flow rate: 900.0
Flowgroup: ring_long_1_4, Flow rate: 900.0

Ring Network with 8 nodes and a shortcut (1-5)
Route for ring_long_1_4 (1-2-3-4): ['1-2', '2-3', '3-4']
Flowgroup: ring_long_1_4, Flow rate: 900.0

Ring Network with 8 nodes and a shortcut (1-5)
Flowgroup: ring_long_1_5, Flow rate: 450.0
Flowgroup: ring_long_1_4, Flow rate: 450.0


In [33]:
#OBJECTIVE: create a network with links, add flows and then call get_route_result_json to chech which should be the route

network = g2.Network([])

bandwidth = 900

## Creation of a network
network = network.add_link(g2.g2_lib_compiled_bindings.Link(str(1), str(2), '1-2', bandwidth))
network = network.add_link(g2.g2_lib_compiled_bindings.Link(str(2), str(3), '2-3', bandwidth))
network = network.add_link(g2.g2_lib_compiled_bindings.Link(str(3), str(4), '3-4', bandwidth))
network = network.add_link(g2.g2_lib_compiled_bindings.Link(str(4), str(1), '4-1', bandwidth))

network = network.add_flowgroup(['1-2', '2-3'], '1-2/2-3', 1)

#flow to compare
# network = network.add_flowgroup(['1-2', '2-3', '3-4'], '1-2/2-3/3-4', 1)

# print(network.get_flow_rate('1-2/2-3/3-4'))

# network = network.remove_flowgroup('1-2/2-3')

route = network.get_route_result_json('1-2/2-3/4', str(1), str(3), 1, num_routes = 10)

print(route)
network = network.add_flowgroup(['1-2', '2-3', '3-4'], '1-2/2-3/3-4', 1)
print(network.get_flow_rate('1-2/2-3/3-4'))


{'0': {'bpg': {'level': 3, 'tree': [{'fairshare': 450.0, 'gradient': 0.0, 'id': '1-2', 'level': 0, 'out_direct': [], 'out_indirect': [], 'type': 'link'}, {'fairshare': 450.0, 'gradient': 0.0, 'id': '2-3', 'level': 0, 'out_direct': [], 'out_indirect': [], 'type': 'link'}]}, 'fgg': {'level': 3, 'tree': [{'gradient': 0.0, 'id': '1-2/2-3', 'level': 1, 'num_flows': 1, 'out': [], 'rate': 450.0, 'type': 'flow'}, {'gradient': 0.0, 'id': '1-2/2-3/4', 'level': 1, 'num_flows': 1, 'out': [], 'rate': 450.0, 'type': 'flow'}, {'fairshare': 450.0, 'gradient': 0.0, 'id': '1-2', 'level': 0, 'out': ['1-2/2-3/4', '1-2/2-3'], 'type': 'link'}, {'fairshare': 450.0, 'gradient': 0.0, 'id': '2-3', 'level': 0, 'out': ['1-2/2-3', '1-2/2-3/4'], 'type': 'link'}, {'fairshare': 'Infinity', 'gradient': 0.0, 'id': '3-4', 'level': 'Infinity', 'out': [], 'type': 'link'}, {'fairshare': 'Infinity', 'gradient': 0.0, 'id': '4-1', 'level': 'Infinity', 'out': [], 'type': 'link'}]}, 'impact': {'fgg_level': {'1-2': 0, '1-2/2-3':